# Capstone Project 1 - Customer Churn Prediction

A complete end-to-end supervised machine learning project.

## Project Workflow

1. Load Dataset
2. Data Cleaning
3. Exploratory Data Analysis
4. Feature Engineering
5. Train-Test Split
6. Model Training
7. Model Evaluation
8. Hyperparameter Tuning
9. Save Best Model
10. Predict New Customers
11. Business Recommendations


In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
import os

df=pd.read_csv("cleaned_data/telco_cleaned.csv")
df.head()

In [ ]:
encoders={}
for c in df.select_dtypes(include="object").columns:
    le=LabelEncoder()
    df[c]=le.fit_transform(df[c].astype(str))
    encoders[c]=le

X=df.drop("Churn",axis=1)
y=df["Churn"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)


## Baseline Random Forest

In [ ]:
rf=RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
pred=rf.predict(X_test)

print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))
print("ROC-AUC:",roc_auc_score(y_test,rf.predict_proba(X_test)[:,1]))


## Hyperparameter Tuning

In [ ]:
params={
    "n_estimators":[100,200],
    "max_depth":[None,10,20],
    "min_samples_split":[2,5]
}

grid=GridSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train,y_train)

best_model=grid.best_estimator_

print("Best Parameters:",grid.best_params_)
print("Best CV Score:",grid.best_score_)


## Final Evaluation

In [ ]:
pred=best_model.predict(X_test)

print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))
print("ROC-AUC:",roc_auc_score(y_test,best_model.predict_proba(X_test)[:,1]))


## Save Model

In [ ]:
os.makedirs("capstone_1/models",exist_ok=True)
joblib.dump(best_model,"capstone_1/models/customer_churn_model.pkl")
print("Model saved.")

## Business Recommendations

- Contact customers with high churn probability.
- Offer personalized discounts.
- Improve support for high-risk segments.
- Monitor churn monthly and retrain the model periodically.


# Conclusion
This capstone delivers a production-ready churn prediction model that can be integrated with a FastAPI backend and Streamlit dashboard.